# Splitting and Chunking
One of the most effective strategies to improve the performance of your language model (LLM) based applications is to _split_ your large date into _smaller chunks_. The goal is to give the LLM only the information it needs for your task **and nothing more!** This practice is the art & science of **Text Splitting**. It is one of the first and most foundational decision that a LLM practitioner will need to make. 

In this notebook, we'll cover the **5 levels of text splitting** that squeeze more performance from your language model applications using the same data that you already have. We'll cover the following techniques:
**Naive Splitting techniques** - focused on physical positional and structure of the source documents. Techniques such as:
1. **Character Splitting** - where you _split_ your documents _by static character limit_.
2. **Recursive Character Text Splitting** - where you start with your long document and recursively go through it and separete it by a set of separators
3. **Document Specific text splitting** - different techiques, like multi-modal, depending on type of documents you have [Python code, Java Code, PDFs with tables & images etc.]

**Advanced Splitting techniques** - which focus on the _contents_ of the documents, beyond just the structure. 
4. **Semantic Splitting** -  Splitting techniques focus on meaning of text chunks
5. **Agentic Splitting** - we'll build a agent that reviews our text and decides how to split it (as Agents have reasoning ability!)

## Why do we need splitting
LLMs are trained on a whole lot of data, but they are not trained on _your data_ (i.e., your organization's data). So while they can answer "worldy" questions, such as "What is the capital of the USA?", they certainly cannot answer questions about your organization's vacation policies, such as "Can I apply for casual leave post-facto?". To answer questions related to _your_ data, you can pass _your_ documents to the LLM and ask it to _ground_ its responses in the _context_ of _your_ data. So, for example, if you feed it all _your_ vaction policy documents, then it would be able to answer the latter question - "Can I apply for casual leav post-facto?". 

However, there is a challenge - you cannot feed entire documents to your LLM - the size of document(s) you can feed is limited by the size of the model's context window. Now doubt, modern LLMs have huge context windows [for example OpenAI 3.5 Turbo has a context size of ~13K tokens (or ~25 pages), GPT 5 Nano has a context window of 256K tokens of ~400 pages!]. Even so, there is a limit on the context size. Also, LLMs perform better when you **increase the signal-to-noise ratio**, meaning _remove information that is not helpful to your task from the context_. So for example, if you are asking about "vacation policies", the context must have information _only_ about vacation policies and nothing else! So even if you _can_ feed newer LLMs a huge amount of data, you should look at _pruning_ the data you feed them for better performance.

**Text splitting (or chunking)** is the process of splitting your data into smaller pieces, so you can make it optimal for your tasks and your language model. You should **always ask yourself**  _"What is the optimal way for me to pass data my language model needs for the task at hand?"_. Our **goal is _not_ to chunk for chunking sake, but rather, to get the data in a format from where responses can be retrieved for value later**. 

In the [Naive RAG Pipeline](01_Naive_RAG_Pipeline.ipynb) notebook, we covered the steps involved i the Naive RAG pipeline. One of the first steps is **Document Loading & Splitting**, which is exactly where **chunking strategies** will apply.

## Character Splitting
This is the most basic form of splitting your text into chunks. It **simply divides your text into N-character sized chunks**, _regardless of their content or form_!
* **Pros:** Easy & Simple
* **Cons:** Very rigid. Does not account for the structure of your text.

Concepts to know:
* **Chunk Size:** the number of characters you'd like in your chunk. 50, 100, 200 etc.
* **Chunk Overlap:** the amount of characters by which you would like consequitive chunks to overlap. This is to ensure that we don't lose information across sentences that straddle chunks. Chunk overlap must be significanly < chunk size. For example 10, 25, 50 etc.


In [17]:
# the text has extra white spaces DELIBERATELY inserted - don't remove them
text = "This is the text I would like to      chunk up. This is    example text for this exercise"

First we'll split manually - it's really very simple!

In [18]:
chunks = []
chunk_size = 35  # no of characters (try with different values!)

for i in range(0, len(text), chunk_size):
    chunk = text[i : i + chunk_size]
    chunks.append(chunk)

chunks

['This is the text I would like to   ',
 '   chunk up. This is    example tex',
 't for this exercise']

Now that was simple, wasn't it? However notice the results - the split is happening blindy at every `chunk_size` interval, which also presents an unpleasant side-effect. The text is splitting up a word right in the middle - that does not make any sense!

In the real language-model world, it's uncommon to work with raw text strings like the above. It's more common to work with documents. Documents are objects that not only hold the text you are concerned with, but also metadata (such as source file, page number etc.) that makes filtering and manipulation easier at a later stage.

The LangChain framework makes it easy to convert our source text into "Documents", specifically the `langchain_text_splitters.CharacterTextSplitter` class, which also addresses the issue highlighted above. Let's see an example of putting this class to use - on the same `text` and using the same `chunk_size`.

In [26]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    chunk_size=35,
    chunk_overlap=5,
    # separator character to split on [blank means split exactly at chunk_size value]
    # if I specify " ", instead, it will not respect the chunk_size value exactly, but split
    # at the nearest to chunk_size space character (and also ignore multiple spaces)
    separator=" ",  # replace with " " and check out the difference
    # strip_whitespace=True -> strip white spaces at beginning or end of chunk of text
    # False = keep white spaces at beginning or end of chunk of text
    strip_whitespace=False,
)

# then split our text as before using create_documents() method of text_splitter
docs = text_splitter.create_documents([text])
docs

[Document(metadata={}, page_content='This is the text I would like to'),
 Document(metadata={}, page_content='to chunk up. This is example text'),
 Document(metadata={}, page_content='text for this exercise')]

In [27]:
for doc in docs:
    print(f"{doc.page_content} -> {len(doc.page_content)}")

This is the text I would like to -> 32
to chunk up. This is example text -> 33
text for this exercise -> 22


Notice that the splits are _not exactly_ `chunk_size` in length. I used " " as separator!, and now the splitting _respects_ the `separator` value (so splits at the nearest to 35 [chunk_size] space charater). Also notice that characters overlap in subsequent chunks due to the `chunk_overlap` parameter.